# Advanced agents — Travel agent (MCP), State & Runtime Context

Three lessons of Module 2, in one notebook. The agent from `2_1_MCP` gains three abilities:
it books a **direct flight** through a public travel MCP server (Kiwi — online, I did not
build it), it keeps more than messages thanks to a **custom state**, and it receives data
for a single run through a **runtime context** object.

Only a Gemini key is needed (from `.env`, Colab Secrets or pasted below): `GOOGLE_API_KEY` —
the local project keeps the same key as `GEMINI_API_KEY`, both are accepted. Same small, cheap
model as every notebook in this path: `gemini-3.1-flash-lite`. The Kiwi MCP server is public,
it needs no key of its own.

> **Model fix:** the original lessons used an OpenAI model (`gpt-5-nano`) which needs an
> OpenAI key. Like every notebook in this path, this one uses Gemini — no extra key, no extra cost.

**Idea:** there are two ways to give an agent information beyond the tools:

- **State** is kept *inside* the agent (checkpointer + thread id, lesson 1_3), and tools can write it back with a `Command`.
- **Runtime context** is *not* saved: the caller passes a typed object at call time, and the tools can read it.

**Lesson plan:**
1. **Travel agent** — connect to Kiwi's public MCP server over HTTP (`streamable_http`) and ask for a direct flight.
2. **Custom state** — my own state fields, written from a tool with `Command`, read back by another tool.
3. **Runtime context** — a typed per-run object, readable from the tools through `runtime.context`.

> Lesson **2_3 (multi-agent)** has its own notebook: `2_3_Multi_agent_kassouar.ipynb`.


## Setup

In [1]:
import importlib.util as _ilu
import subprocess as _sp
import sys as _sys

def _check_install(pkg: str, mod: str):
    if _ilu.find_spec(mod) is None:
        result = _sp.run([_sys.executable, "-m", "pip", "install", "-q", pkg],
                         capture_output=True, text=True)
        print(f"{pkg}: installed" if result.returncode == 0
              else f"{pkg}: install FAILED -> {result.stderr[-200:]}")
    else:
        print(f"{pkg}: already installed")

for _pkg, _mod in [
    ("python-dotenv", "dotenv"),
    ("langchain", "langchain"),
    ("langgraph", "langgraph"),
    ("langchain-mcp-adapters", "langchain_mcp_adapters"),
    ("google-genai", "google.genai"),
]:
    _check_install(_pkg, _mod)

print("Setup done.")

python-dotenv: already installed
langchain: already installed
langgraph: already installed
langchain-mcp-adapters: already installed
google-genai: already installed
Setup done.


In [2]:
import os
from getpass import getpass
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

# Load the local .env file: it lives in the notebooks folder (CWD when the
# notebook is launched by run_notebook.cmd) or one folder up (module root).
# override=True makes the file win over any OS-variable key.
_dotenv = Path.cwd() / ".env"
if not _dotenv.exists():
    _dotenv = Path.cwd().parent / ".env"

if _dotenv.exists():
    load_dotenv(_dotenv, override=True)
    _key_source = f"folder .env ({_dotenv.parent.name}/)"
else:
    load_dotenv(find_dotenv(), override=True)
    _key_source = "OS variable or prompt"

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass(f"Please paste your {var} (hidden): ")

_set_env("GOOGLE_API_KEY")
_set_env("GEMINI_API_KEY")

print("Key source:", _key_source)
print("GOOGLE_API_KEY:", "set" if os.environ.get("GOOGLE_API_KEY") else "MISSING")
print("GEMINI_API_KEY:", "set" if os.environ.get("GEMINI_API_KEY") else "MISSING")
print("Setup done.")

Key source: folder .env (notebooks/)
GOOGLE_API_KEY: set
GEMINI_API_KEY: set
Setup done.


In [3]:
# Same small, cheap Gemini model as every notebook in this path (Module 1 + 2_1).
MODEL = "google_genai:gemini-3.1-flash-lite"
print("Model:", MODEL)

Model: google_genai:gemini-3.1-flash-lite


## Part 1: Travel agent — a public MCP server (HTTP)

Lesson `2_1_MCP` connected the agent to a *local* server over `stdio`. Here the same client
points at a **public** server: Kiwi.com exposes its flight search as an MCP endpoint served
over HTTP. I did not write the server, and it needs no key.

> **`transport: "streamable_http"`**: the transport used for online MCP servers — the client and
> the server talk over HTTP instead of launching a subprocess. Everything else stays the same.

### Task 1: connect to the server and list its tools

In [4]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "travel_server": {
            "transport": "streamable_http",
            "url": "https://mcp.kiwi.com",
        }
    }
)

tools = await client.get_tools()
print(f"Tools exposed by the travel server ({len(tools)}):")
for t in tools:
    first_line = ((t.description or "").strip().splitlines() or [""])[0]
    print(f" - {t.name}: {first_line[:120]}")

Tools exposed by the travel server (2):
 - search-flight: # Search for flights
 - feedback-to-devs: Send feedback, bug reports, or feature requests about the Kiwi.com search-flight MCP tool to its developers. This channe


In [5]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    MODEL,
    tools=tools,
    checkpointer=InMemorySaver(),
    system_prompt="You are a travel agent. No follow up questions.",
)

print("Travel agent ready.")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Travel agent ready.


### Task 2: ask for a direct flight

The agent must understand the request, call `search-flight` with the right arguments (its
schema tells it what the server expects), and turn the answer into a sentence. Same thread id
pattern as lesson 1_3 / 2_1, thanks to the checkpointer.

In [6]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Get me a direct flight from San Francisco to Tokyo on March 31, 2027")]},
    config,
)

print(response["messages"][-1].content)

[{'type': 'text', 'text': "Here are the available options for your flight from San Francisco (SFO) to Tokyo (TYO) on March 31, 2027.\n\n### Direct Flights\n\n| Route | Times & duration | Cabin | Price | Booking Link |\n| :--- | :--- | :--- | :--- | :--- |\n| SFO → NRT | 31 Mar 12:20 → 01 Apr 15:25 (11h 05m) | Economy | 448 EUR | [Book Now](https://kiwi.com/u/2qg756) |\n\n***\n\n### Summary\n*   **Best Price & Shortest Option:** The direct flight with All Nippon Airways is both the cheapest and fastest option available.\n\nI recommend booking the direct flight with All Nippon Airways to save both time and money. Have a wonderful trip to Tokyo!\n\n**Fun Fact:** Tokyo is home to the world's busiest pedestrian crossing, the Shibuya Crossing, where as many as 3,000 people can cross the street at once during peak times!", 'extras': {'signature': 'EnMKcQFpFH0TXZd39zXyOkhOhdmGrz2JzNkaa3DBdigDpLfQ5W1pTnqKC6CTRmD1Zn71Z+wvzU2l59h5ZrLgkFFYoHhYyppHLN4z8LVmdwg6VoP/KQoTKl2wxaVBzoxPWb9T/LeF17NZGtwQI10

In [7]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Get me a direct flight from San Francisco to Tokyo on March 31, 2027', additional_kwargs={}, response_metadata={}, id='e0763eaa-08d9-4d27-a624-f101eac57fb8'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search-flight', 'arguments': '{"departureDate": "31/03/2027", "flyFrom": "SFO", "flyTo": "TYO"}'}, '__gemini_function_call_thought_signatures__': {'call_233384': 'EnMKcQFpFH0TJZjNO3MJY3kBGxvDsNx/pNkzCb3tJQpVCf0IDlWXYtZW3KDEADYdBMEh0Yn7WKNOGeyNq1Rfd+FuiFr8cCD+07MtU+/JeAXb2BExg0kO4SLmSJZV6jwLh/Chhizd5GijUoXsbvlRKQfFgo5R'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0dacb-1572-7912-8b1b-5a4c2a241bea-0', tool_calls=[{'name': 'search-flight', 'args': {'departureDate': '31/03/2027', 'flyFrom': 'SFO', 'flyTo': 'TYO'}, 'id': 'call_233384', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tok

I never wrote a flight-search API call: the tool, the server and the HTTP transport did the
whole trip. Later, changing the server only means editing that one `travel_server` entry — the
agent code did not change at all.

## Part 2: A custom state (beyond messages)

Lesson 1_3 showed the state as the list of messages, saved by a checkpointer. Here the state
holds its **own field** — `favourite_colour`. A tool writes it into the state with a `Command`,
another tool reads it back.

> **`AgentState`**: the default state (it holds `messages`). Subclass it to add fields.

> **`runtime: ToolRuntime`**: a parameter a tool can declare to reach the running agent —
> `runtime.tool_call_id`, `runtime.state` and `runtime.context`.

> **`Command(update={...})`**: what a tool returns to *change* the state. The agent applies it
> as the answer of the tool.

### Task 1: define the state and a writing tool

In [8]:
from langchain.agents import AgentState

class CustomState(AgentState):
    favourite_colour: str

print("CustomState fields:", list(CustomState.__annotations__))

CustomState fields: ['messages', 'jump_to', 'structured_response', 'favourite_colour']


In [9]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(update={
        "favourite_colour": favourite_colour,
        "messages": [ToolMessage("Successfully updated favourite colour", tool_call_id=runtime.tool_call_id)],
    })

print("update_favourite_colour ready.")

update_favourite_colour ready.


In [10]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    MODEL,
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState,
)

print("Agent with custom state ready.")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Agent with custom state ready.


In [11]:
from langchain.messages import HumanMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}},
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='d5744731-ed9b-4241-8074-455d8b3c9a69'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'update_favourite_colour', 'arguments': '{"favourite_colour": "green"}'}, '__gemini_function_call_thought_signatures__': {'call_156027': 'EnMKcQFpFH0T23cILGIFf6YJ4i2a7qsJ/aZfGFtsJEeFN9yeLu+xWZsMN9YE9blgHIDvlb/MEEXsfU5I0/Tt8dCb0DZyVxlEHR6Gel/hFRBDqGlm8BQ8JaEOAM0A2LLkoW+8aJfniG1+sjH7rVJtH2E+06O+'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0dacb-4f51-7e70-8f52-b9fd48e9600e-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'call_156027', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 69, 'output_tokens': 22, 'total_tokens': 91, 'input_toke

The tool's `Command` wrote the field **into the state**, even though it never appears in the
*messages*: check `favourite_colour` in the final state below.

In [12]:
print("favourite_colour in the final state:", response["favourite_colour"])

favourite_colour in the final state: green


### Task 2: seed the state at call time

The state can also be seeded directly by the caller. This run uses a **new thread id** (`"10"`),
so nothing is loaded from memory — the state starts from the dict passed to `invoke`.

In [13]:
response = agent.invoke(
    {
        "messages": [HumanMessage(content="Hello, how are you?")],
        "favourite_colour": "green",
    },
    {"configurable": {"thread_id": "10"}},
)

pprint(response)
print("favourite_colour:", response["favourite_colour"])

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}, id='43afd131-abb3-4106-9945-26c5f9513d0f'),
              AIMessage(content=[{'type': 'text', 'text': "I'm doing well, thank you for asking! How are you today?", 'extras': {'signature': 'EnMKcQFpFH0THyLHMRTPwm8mUx7U2DD0ANQr6mtbJeTQ/WP/fEfRpFBcSdwDW76J9HjF3PWjBlzVTHKPQPr0FPhi3Qx1wo0tiYU16w0tn8ePywyJ3FZM1pvGzzasUv4LbH0sIxxPmqnTaNrOn6w1Zfgv5ikT'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0dacb-5fd7-70d2-9719-b1a17139a777-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 70, 'output_tokens': 16, 'total_tokens': 86, 'input_token_details': {'cache_read': 0}})]}
favourite_colour: green


### Task 3: read the state back

A second tool that *reads* the state. If the field is absent (a fresh thread), it says so
instead of failing.

In [14]:
@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"

agent = create_agent(
    MODEL,
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState,
)

print("Agent with read tool ready.")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Agent with read tool ready.


In [15]:
# Write it first (thread "1"), then ask on the same thread — the tool reads it back.
response = agent.invoke(
    {"messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}},
)

pprint(response)

{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='3e793245-fc27-45eb-8d7e-f7e87f069ec5'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'update_favourite_colour', 'arguments': '{"favourite_colour": "green"}'}, '__gemini_function_call_thought_signatures__': {'call_247309': 'EnMKcQFpFH0T1K9ET+TFUnkZBFUWmjo1LJ5HDIyIMKkVj0w9TbsVEtmLpKdHvd/NQv9scNXpA6R0JKa//T18/jBd5IBHPr84vqI7E/s9hGQOI3YZJ31vdeWT6c/ao/JfSGU3abZRW5OhNy0dllu39oKMWaRV'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0dacb-6a69-7cb0-9a5d-26f2aeb64e51-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'green'}, 'id': 'call_247309', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 96, 'output_tokens': 22, 'total_tokens': 118, 'input_tok

In [16]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "1"}},
)

print(response["messages"][-1].content)
pprint(response)

[{'type': 'text', 'text': 'Your favourite colour is green!', 'extras': {'signature': 'EnMKcQFpFH0ToC2lSImxKBGpwhWp5lnn5+wUca1wydnOc1q2jiMRND5drJp6/By3+7ULCy7z8Ga5d0wshnmejhNqC3HoGBtZaQu8mZIaDk1VBDzGvA15SUDbjptIiIeIcIM492Sc2hMpGz4YFxjGbjXIoPf+'}}]
{'favourite_colour': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='3e793245-fc27-45eb-8d7e-f7e87f069ec5'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'update_favourite_colour', 'arguments': '{"favourite_colour": "green"}'}, '__gemini_function_call_thought_signatures__': {'call_247309': 'EnMKcQFpFH0T1K9ET+TFUnkZBFUWmjo1LJ5HDIyIMKkVj0w9TbsVEtmLpKdHvd/NQv9scNXpA6R0JKa//T18/jBd5IBHPr84vqI7E/s9hGQOI3YZJ31vdeWT6c/ao/JfSGU3abZRW5OhNy0dllu39oKMWaRV'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0dacb-6a69-7cb0-9a5d-26f2aeb64e51

## Part 3: Runtime context

Some data describes the run, not the conversation, and should not be saved anywhere: a language,
a user preference... The **runtime context** is a typed object passed by the caller at call time.
The model sees it only if a tool reads it through `runtime.context`.

> **`context_schema=`**: the argument of `create_agent` that tells the agent the type of the context object.

> **`agent.invoke(..., context=MyContext())`**: passes the context for this run only — nothing is persisted.

### Task 1: a context the model can reason about

In [17]:
from dataclasses import dataclass

@dataclass
class ColourContext:
    favourite_colour: str = "blue"
    least_favourite_colour: str = "yellow"

print(ColourContext())

ColourContext(favourite_colour='blue', least_favourite_colour='yellow')


In [18]:
from langchain.agents import create_agent

agent = create_agent(
    model=MODEL,
    context_schema=ColourContext,
)

print("Agent with context_schema ready.")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Agent with context_schema ready.


### Task 2: ask with a context

In [19]:
from langchain.messages import HumanMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext(),
)

pprint(response)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='3dc553b0-031b-4c1b-b532-699bddb43e0f'),
              AIMessage(content=[{'type': 'text', 'text': 'I don’t know what your favourite colour is yet, as I don’t have access to your personal history or thoughts. \n\nIf you tell me, I’ll be sure to remember it! So, what is it?', 'extras': {'signature': 'EnMKcQFpFH0TfuIn9UTWgkS9VO/ps65d4EaucX9dF6RWQYwu5pke4mk8ttiTeIjS+5awPYpJqwc5Zt8TGzfF5xBVijZK6SMGrO58jHcYcDizKiO7du7VhDG8eVsV1T28e0vdOoGZQQuiYMKGwttpTdQloERV'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0dacb-8339-75d0-9997-f90fb2a27eb6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 7, 'output_tokens': 48, 'total_tokens': 55, 'input_token_details': {'cache_read': 0}})]}


### Task 3: access the context from a tool

Now the context becomes visible to the model: two tools read `runtime.context` directly, typed
to the schema with `ToolRuntime[ColourContext]`.

In [20]:
from langchain.tools import tool, ToolRuntime

@tool
def get_favourite_colour(runtime: ToolRuntime[ColourContext]) -> str:
    """Get the favourite colour of the user"""
    return runtime.context.favourite_colour

@tool
def get_least_favourite_colour(runtime: ToolRuntime[ColourContext]) -> str:
    """Get the least favourite colour of the user"""
    return runtime.context.least_favourite_colour

print("Context tools ready.")

Context tools ready.


In [21]:
agent = create_agent(
    model=MODEL,
    tools=[get_favourite_colour, get_least_favourite_colour],
    context_schema=ColourContext,
)

print("Agent with context tools ready.")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Agent with context tools ready.


In [22]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext(),
)

pprint(response)

{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='f5c74c1a-f411-4cd5-a82b-196d655abf45'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_favourite_colour', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'call_284116': 'EnMKcQFpFH0TYqkENnAPpMdFbaE5mDdTia3xsCp0tIyI2I5/OFLPNYBC8S2VoWkJ8BkWW4SRuTPKY24+q75WXgRQfwnJwfFBpJG7eGG5a+GtrzPDn5xqGqSSoORvzezv7sf0e5LJOc1ZA3eVX1VEL74z2yM2'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0dacb-88be-7d40-9cbe-84c21f4be352-0', tool_calls=[{'name': 'get_favourite_colour', 'args': {}, 'id': 'call_284116', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 56, 'output_tokens': 13, 'total_tokens': 69, 'input_token_details': {'cache_read': 0}}),
              ToolMessage(content='blue', name='get_favou

In [23]:
# A different context for this run only: the favourite colour becomes green.
response = agent.invoke(
    {"messages": [HumanMessage(content="What is my favourite colour?")]},
    context=ColourContext(favourite_colour="green"),
)

print(response["messages"][-1].content)
pprint(response)

[{'type': 'text', 'text': 'Your favourite colour is green.', 'extras': {'signature': 'EnMKcQFpFH0T+wpLRpK1koKDX+mfY9wErYo5GwTte5ckNYfqzw7tw5Jb3tlrCNGcihNlAKjpV5G9dw7AYAWyg78GE+nP5avwMR5vavyhLvtKf2RtJ5wurY9UCPUY/b/td6Tn19015g6xzOF1XI/5nygrOJ0P'}}]
{'messages': [HumanMessage(content='What is my favourite colour?', additional_kwargs={}, response_metadata={}, id='8b8ccacd-0377-4155-b740-08cd7aaa3cd6'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_favourite_colour', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'call_388642': 'EnMKcQFpFH0T4B+kYXZ5U+eP+qZo/uQedCEbLnA2p9xFS70vaCgFcxLdWiT0qmuWqajq9ruJnbGgof18IlR3lLlgqqon9CQ6zHyOVxUu+jA32s5NEKn3XTuWmArjauNTheHsEUx4jPPTSliKrUPfNWm4iaFa'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0dacb-9005-7a43-9459-b9daceea6b01-0', tool_calls=[{'name': 'get_favourite_colour', 'args': {}

## What I learned

1. **A public MCP server is an instant toolbox.** Kiwi's flight search arrived over HTTP (`streamable_http`) with zero setup, and the agent called it on its own.
2. **The state is a schema, not just a list of messages.** Subclass `AgentState`, add a field, and a tool can write it with `Command(update={...})`.
3. **A tool can reach the running agent** through `runtime: ToolRuntime` — `runtime.tool_call_id`, `runtime.state` and `runtime.context` are all available.
4. **Not everything belongs in the state.** The runtime context is passed per call, typed with `context_schema`, and read from tools with `runtime.context`. Nothing is saved.
5. **State vs context:** the state is the memory the agent keeps (thread id + checkpointer); the context is data the caller gives for exactly one run.


## Summary of functions

| Function / class | Simple definition |
|---|---|
| `MultiServerMCPClient({name: {transport: "streamable_http", url}})` | Connects the agent to a public MCP server over HTTP. |
| `await client.get_tools()` | Asks the server(s) for their tools. |
| `create_agent(MODEL, tools=..., checkpointer=...)` | The same agent factory as Module 1. |
| `class MyState(AgentState)` | Adds my own fields to the agent's state. |
| `@tool` + `runtime: ToolRuntime` | A tool that can touch the running agent (`state`, `context`, `tool_call_id`). |
| `Command(update={...})` | Returned by a tool to write state fields (and messages). |
| `create_agent(..., state_schema=MyState)` | Tells the agent to use the custom state class. |
| `@dataclass MyContext` | A typed object describing a single run. |
| `create_agent(..., context_schema=MyContext)` | Tells the agent the type of the run context. |
| `agent.invoke(..., context=MyContext(...))` | Passes the context for this run only. |


By
> Kassouar Fatima                   26-09-2026
